In [ ]:
# Installs take ~1 min on Colab
!pip install transformers torch torchvision Pillow opencv-python-headless matplotlib -q

import torch
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {'GPU ✅  ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (slower but works)'}")


In [ ]:
import os

# ── Option A: Colab file upload ──────────────────────────────────────────
try:
    from google.colab import files
    print("📎 Select your image file...")
    uploaded = files.upload()
    IMAGE_PATH = list(uploaded.keys())[0]
    print(f"✅ Uploaded: {IMAGE_PATH}")
except ImportError:
    # ── Option B: local path ─────────────────────────────────────────────
    IMAGE_PATH = "1773508687566_41.png"   # ← change this
    print(f"📁 Using local file: {IMAGE_PATH}")

OUTPUT_DIR = "trocr_char_crops"
os.makedirs(OUTPUT_DIR, exist_ok=True)

from PIL import Image
import matplotlib.pyplot as plt

orig = Image.open(IMAGE_PATH).convert("RGB")
plt.figure(figsize=(12, 5))
plt.imshow(orig); plt.axis('off')
plt.title(f"Input image: {orig.size[0]}×{orig.size[1]}px", fontsize=12)
plt.tight_layout(); plt.show()


# Step 2 — Load TrOCR (handwritten model)

`microsoft/trocr-base-handwritten` is a **ViT encoder + RoBERTa decoder**:
- **Encoder**: BEiT ViT splits image into 16×16 patches → 196 patch embeddings
- **Decoder**: autoregressively generates characters, cross-attending to those patches

We enable `output_attentions=True` so we can read the cross-attention weights.


In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on {device}...")

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model     = VisionEncoderDecoderModel.from_pretrained(
                "microsoft/trocr-base-handwritten").to(device)
model.eval()

print("✅ Model loaded!")
print(f"   Encoder : {model.encoder.__class__.__name__}")
print(f"   Decoder : {model.decoder.__class__.__name__}")
print(f"   Params  : {sum(p.numel() for p in model.parameters())/1e6:.0f}M")


# Step 3 — Segment into text lines

TrOCR works on **one line at a time** (like most OCR models).  
We use OpenCV projection to find the horizontal text-line bands.  
Each line crop is fed to TrOCR independently.


In [ ]:
import cv2
import numpy as np

img_rgb = np.array(orig)
gray    = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
H, W    = gray.shape

# Binary image (white text on black)
binary  = cv2.adaptiveThreshold(gray, 255,
              cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, -5)

# Row projection → find line bands
row_proj = binary.mean(axis=1)
LINE_T   = 20.0
MIN_H    = 20

in_line, start, line_bands = False, 0, []
for y in range(H):
    if not in_line and row_proj[y] > LINE_T:
        in_line = True; start = y
    elif in_line and row_proj[y] <= LINE_T:
        in_line = False
        if y - start > MIN_H:
            line_bands.append((max(0, start-4), min(H, y+4)))
if in_line:
    line_bands.append((max(0, start-4), H))

print(f"Lines detected: {len(line_bands)}")
for i, (a, b) in enumerate(line_bands):
    print(f"  Line {i+1}: rows {a}–{b}  ({b-a}px tall)")

# Visualise
fig, axes = plt.subplots(1, len(line_bands)+1, figsize=(16, 4))
axes[0].imshow(img_rgb); axes[0].axis('off'); axes[0].set_title("Full image")
colors = [(255,80,80),(80,255,130),(80,180,255),(255,220,50)]
vis = img_rgb.copy()
for i,(a,b) in enumerate(line_bands):
    col = colors[i%len(colors)]
    cv2.rectangle(vis,(0,a),(W-1,b),col,2)
    axes[i+1].imshow(img_rgb[a:b,:])
    axes[i+1].axis('off')
    axes[i+1].set_title(f"Line {i+1}")
axes[0].imshow(vis); axes[0].axis('off')
plt.suptitle("Detected text lines", fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


# Step 4 — TrOCR inference + extract cross-attention maps

For each line:
1. Feed the line image to TrOCR
2. Decode tokens one by one (each = one character or word-piece)
3. At each decoding step, **read the cross-attention weights** from the last decoder layer
4. The cross-attention shape is `(num_heads, 1, num_patches)` → mean over heads → reshape to `(14, 14)` patch grid
5. Upsample to the line's pixel dimensions → this is our **character heatmap**
6. Find the peak region → bounding box → crop

This is exactly how TrOCR "sees" each character.


In [ ]:
from transformers.generation import GenerateDecoderOnlyOutput
import torch.nn.functional as F

def decode_with_attention(line_img_pil, device, processor, model):
    """
    Run TrOCR on a single line image.
    Returns:
        tokens     : list of decoded character strings
        attn_maps  : list of (H_line, W_line) numpy arrays — one per token
        text       : full decoded string
    """
    # Preprocess
    pixel_values = processor(images=line_img_pil, return_tensors="pt").pixel_values.to(device)

    # Greedy decode with attention output
    with torch.no_grad():
        outputs = model.generate(
            pixel_values,
            output_attentions=True,
            return_dict_in_generate=True,
            max_new_tokens=64,
        )

    generated_ids = outputs.sequences[0]
    text = processor.tokenizer.decode(generated_ids, skip_special_tokens=True)

    # cross_attentions: tuple[steps] of tuple[layers] of tensor(batch, heads, tgt_len, src_len)
    # src_len = 197 (196 patches + 1 CLS token) for 224x224 ViT
    cross_attns = outputs.cross_attentions   # tuple of (num_steps,)

    lw, lh = line_img_pil.size   # line width, height in pixels
    patch_grid = 14              # ViT base: 14×14 = 196 patches

    tokens   = []
    attn_maps = []

    for step_idx, step_attns in enumerate(cross_attns):
        # step_attns: tuple of num_layers tensors, shape (1, heads, 1, 197)
        last_layer = step_attns[-1]   # (1, heads, 1, 197)
        attn = last_layer[0].mean(0)  # (1, 197) — mean over heads
        attn = attn[0, 1:]            # (196,) — drop CLS token

        # Reshape to patch grid
        patch_map = attn.reshape(patch_grid, patch_grid).cpu().float().numpy()

        # Normalize
        patch_map = (patch_map - patch_map.min()) / (patch_map.max() - patch_map.min() + 1e-8)

        # Upsample to line pixel dimensions
        heatmap = cv2.resize(patch_map, (lw, lh), interpolation=cv2.INTER_CUBIC)
        attn_maps.append(heatmap)

        # Decode token
        tok_id = generated_ids[step_idx + 1].item() if step_idx + 1 < len(generated_ids) else 0
        tok_str = processor.tokenizer.decode([tok_id], skip_special_tokens=True)
        tokens.append(tok_str)

    return tokens, attn_maps, text


# ── Run on all lines ──────────────────────────────────────────────────────
all_results = []   # list of (line_idx, tokens, attn_maps, decoded_text, y0, y1)

for li, (y0, y1) in enumerate(line_bands):
    line_crop = Image.fromarray(img_rgb[y0:y1, :])
    print(f"\nLine {li+1}  ({y1-y0}px tall)  →  decoding...", end=" ")
    tokens, attn_maps, text = decode_with_attention(line_crop, device, processor, model)
    all_results.append((li, tokens, attn_maps, text, y0, y1))
    print(f'"{text}"  ({len(tokens)} tokens)')


# Step 5 — Visualise cross-attention maps

For each line, plot the cross-attention heatmap for every decoded token.  
High-activation regions = where the model was looking when it generated that character.


In [ ]:
import matplotlib.cm as cm

for li, tokens, attn_maps, text, y0, y1 in all_results:
    line_img = img_rgb[y0:y1, :]
    n = len(tokens)
    if n == 0:
        continue

    ncols = min(12, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*1.6, nrows*2.2))
    fig.patch.set_facecolor('#0d0d1a')
    fig.suptitle(f'Line {li+1}: "{text}" — cross-attention per token',
                 fontsize=11, fontweight='bold', color='white')
    axes_flat = np.array(axes).flatten() if n > 1 else [axes]

    for i, (tok, amap) in enumerate(zip(tokens, attn_maps)):
        overlay = line_img.copy().astype(np.float32) / 255.0
        heat    = cm.inferno(amap)[:, :, :3]
        blended = (0.45 * overlay + 0.55 * heat).clip(0, 1)
        axes_flat[i].imshow(blended)
        axes_flat[i].set_title(repr(tok), fontsize=8, color='#ffdd80', pad=2)
        axes_flat[i].axis('off')

    for j in range(n, len(axes_flat)):
        axes_flat[j].axis('off')

    plt.tight_layout()
    plt.show()
    print()


# Step 6 — Extract character bounding boxes from attention

For each token's attention map:
1. Threshold at 50th percentile (top-half activations)
2. Find the bounding box of the high-attention region
3. Map back to full-image coordinates
4. Save the crop


In [ ]:
from PIL import Image as PILImage
import json

ATTN_THRESHOLD = 0.50   # keep top 50% of activations — lower = larger boxes
PADDING        = 4      # px padding around each crop
MIN_CHAR_W     = 6
MIN_CHAR_H     = 10

saved_chars = []
metadata    = []
total = 0

for li, tokens, attn_maps, text, y0_line, y1_line in all_results:
    lh = y1_line - y0_line
    lw = W

    for ti, (tok, amap) in enumerate(zip(tokens, attn_maps)):
        if not tok.strip():
            continue   # skip whitespace tokens

        # Threshold
        thresh_val = np.percentile(amap, ATTN_THRESHOLD * 100)
        mask       = (amap >= thresh_val).astype(np.uint8) * 255

        # Find bounding box of high-attention region
        coords = cv2.findNonZero(mask)
        if coords is None:
            continue
        rx, ry, rw, rh = cv2.boundingRect(coords)

        # Minimum size filter
        if rw < MIN_CHAR_W or rh < MIN_CHAR_H:
            continue

        # Map to full image coordinates
        abs_x0 = max(0, rx - PADDING)
        abs_y0 = max(0, y0_line + ry - PADDING)
        abs_x1 = min(W, rx + rw + PADDING)
        abs_y1 = min(H, y0_line + ry + rh + PADDING)

        crop  = img_rgb[abs_y0:abs_y1, abs_x0:abs_x1]
        safe  = tok.strip() if tok.strip().isalnum() else f"ord{ord(tok.strip()[0]) if tok.strip() else 0}"
        fname = f"line{li+1:02d}_tok{ti:03d}_{safe}.png"
        PILImage.fromarray(crop).save(f"{OUTPUT_DIR}/{fname}")

        saved_chars.append((tok, fname, li+1, ti, (abs_x0, abs_y0, abs_x1, abs_y1)))
        metadata.append({
            "file": fname, "token": tok,
            "line": li+1, "token_idx": ti,
            "bbox": [int(abs_x0), int(abs_y0), int(abs_x1), int(abs_y1)]
        })
        total += 1

with open(f"{OUTPUT_DIR}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Saved {total} character crops → '{OUTPUT_DIR}/'")
print(f"\nAll tokens extracted:")
for li, tokens, _, text, _, _ in all_results:
    print(f"  Line {li+1}: {repr(text)}")
    print(f"          tokens: {[repr(t) for t in tokens if t.strip()]}")


# Step 7 — Visualise bounding boxes on original image

Show all extracted character crops overlaid on the original image,  
coloured by line.


In [ ]:
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
LINE_COLORS = ['#4fc3f7','#69f0ae','#ff8a65','#ce93d8','#fff176']

# Left: attention bbox overlay on original
axes[0].imshow(img_rgb)
for (tok, fname, li, ti, (x0,y0,x1,y1)) in saved_chars:
    col = LINE_COLORS[(li-1) % len(LINE_COLORS)]
    axes[0].add_patch(mpatches.Rectangle((x0,y0),x1-x0,y1-y0,
                      lw=1.5,edgecolor=col,facecolor='none',alpha=0.9))
    if tok.strip():
        axes[0].text(x0,max(y0-2,0),tok,fontsize=7,color='yellow',fontweight='bold')
axes[0].set_title(f"TrOCR cross-attention boxes ({total} characters)", fontsize=12, fontweight='bold')
axes[0].axis('off')

# Right: contact sheet
n = len(saved_chars)
ncols = min(20, n)
nrows = (n + ncols - 1) // ncols

contact = np.zeros((nrows * 45, ncols * 35, 3), dtype=np.uint8)
for i, (tok, fname, li, ti, (x0,y0,x1,y1)) in enumerate(saved_chars):
    r, c = i // ncols, i % ncols
    crop = img_rgb[y0:y1, x0:x1]
    if crop.size > 0:
        thumb = cv2.resize(crop, (35, 45))
        contact[r*45:(r+1)*45, c*35:(c+1)*35] = thumb

axes[1].imshow(contact)
axes[1].set_title(f"Contact sheet — {n} crops", fontsize=12, fontweight='bold')
axes[1].axis('off')

plt.suptitle("TrOCR Cross-Attention Character Segmentation", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


# Step 8 — Full contact sheet with token labels

In [ ]:
n = len(saved_chars)
ncols = min(20, n)
nrows = (n + ncols - 1) // ncols

fig, axes2 = plt.subplots(nrows, ncols, figsize=(ncols * 1.1, nrows * 1.9))
fig.patch.set_facecolor('#0d0d1a')
fig.suptitle(f"All {n} TrOCR-segmented characters (token = 1 character)",
             fontsize=13, fontweight='bold', color='white')
axes2_flat = np.array(axes2).flatten() if n > 1 else [axes2]

for i, (tok, fname, li, ti, bbox) in enumerate(saved_chars):
    crop = PILImage.open(f"{OUTPUT_DIR}/{fname}")
    axes2_flat[i].imshow(crop)
    axes2_flat[i].set_title(repr(tok), fontsize=7, color='#aaddff', pad=1)
    axes2_flat[i].axis('off')
for j in range(n, len(axes2_flat)):
    axes2_flat[j].axis('off')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/contact_sheet_trocr.png", dpi=150, bbox_inches='tight',
            facecolor='#0d0d1a')
plt.show()
print(f"\n✅ Done! {n} characters saved to '{OUTPUT_DIR}/'")
